# Import libraries

In [10]:
import pandas as pd
import psycopg2
from tqdm import tqdm
import os
from underthesea import word_tokenize, text_normalize, pos_tag
from datetime import datetime
import string
import regex as re
from dotenv import load_dotenv
from sqlalchemy import create_engine, Table, MetaData, update, bindparam
load_dotenv()

True

# Connect to database


## Get database information


In [2]:
postgres_user = os.getenv("POSTGRES_USER")
postgres_password = os.getenv("POSTGRES_PASSWORD")
postgres_host = os.getenv("POSTGRES_HOST")
postgres_port = os.getenv("POSTGRES_PORT")
postgres_db = os.getenv("POSTGRES_DB")

In [3]:
try:
    conn = psycopg2.connect(
        database=postgres_db,
        user=postgres_user,
        host=postgres_host,
        password=postgres_password,
        port=postgres_port,
    )
    print("Opened database successfully")
except Exception as e:
    print(f"Connection failed: {e}")

Opened database successfully


# Get news from database

In [4]:
engine = create_engine(
    f"postgresql://{postgres_user}:{postgres_password}@{postgres_host}:{postgres_port}/{postgres_db}"
)

In [5]:
news_query = """SELECT title, "rawContent", company FROM news"""
news_info = pd.read_sql(news_query, engine)

# Create the dictionary
news_dict = {
    "title": news_info["title"].tolist(),
    "rawContent": news_info["rawContent"].tolist(),
    "company": news_info["company"].tolist(),
}

# Clean data

In [6]:
stop_words = set()
with open("./data/stopwords.txt", "r", encoding="utf-8") as f:
    for line in f:
        stop_words.add(line.strip())

In [7]:
unwanted_tags = ['Np', 'R', 'M', 'CH', 'E', 'Nu', 'C', 'L', 'P']

In [8]:
# Precompile regex patterns
repeat_char_pattern = re.compile(r"([a-z]+?)\1+")
space_punctuation_pattern = re.compile(r"(\w)\s*([" + re.escape(string.punctuation) + "])\s*(\w)")
end_space_punctuation_pattern = re.compile(r"(\w)\s*([" + re.escape(string.punctuation) + "])")
consecutive_punctuation_pattern = re.compile(f"([{re.escape(string.punctuation)}])([{re.escape(string.punctuation)}])+")
number_pattern = re.compile(r'\d+')
multiple_spaces_pattern = re.compile(r"\s+")

In [ ]:
def clean_text(text):    
    # Reduce repeated character (e.g. 'aaabbb' -> 'ab')
    text = repeat_char_pattern.sub(r"\1", text)

    # Ensure space before and after any punctuation mark
    text = space_punctuation_pattern.sub(r"\1 \2 \3", text)
    text = end_space_punctuation_pattern.sub(r"\1 \2", text)

    # Reduce consecutive punctuation
    text = consecutive_punctuation_pattern.sub(r"\1", text)

    # Remove numbers
    text = number_pattern.sub('', text)

    # Remove any leading or trailing spaces, or leading or trailing punctuation marks from the text
    text = text.strip()

    while text.endswith(tuple(string.punctuation + string.whitespace)):
        text = text[:-1]

    while text.startswith(tuple(string.punctuation + string.whitespace)):
        text = text[1:]

    # Remove all punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Reduce multiple spaces
    text = multiple_spaces_pattern.sub(" ", text)

    # POS tagging to remove unwanted words
    pos_tags = pos_tag(text)
    text = [word for word, tag in pos_tags if tag not in unwanted_tags]
    text = " ".join(text)

    # Lowercase text
    text = text.lower()

    # Make sure punctuation is in the right letter (Vietnamese case)
    text = text_normalize(text)

    # Tokenize the cleaned text
    text = word_tokenize(text, format="text")

    text = text.split(" ")

    # Remove stop words
    text = [word for word in text if word.replace("_", " ") not in stop_words]

    # Reduce multiple spaces
    text = [word for word in text if word]

    text = " ".join(text)

    return text

In [ ]:
# Define metadata and table object
metadata = MetaData()
news_table = Table('news', metadata, autoload_with=engine)

# Define the update statement with non-conflicting parameter names
stmt = update(news_table).where(news_table.c.title == bindparam('b_title')).values(cleanedContent=bindparam('b_cleanedContent'))

data = []
news_dict["cleanedContent"] = []
batch_size = 1000

count = 0

for i in tqdm(range(len(news_dict["rawContent"]))):
    sentence = news_dict["rawContent"][i]
    cleaned_sentence = clean_text(sentence)
    news_dict["cleanedContent"].append(cleaned_sentence)
    data.append({"b_cleanedContent": cleaned_sentence, "b_title": news_dict["title"][i]})
    count += 1

    if count % batch_size == 0:
        with engine.connect() as conn:
            print(f"Inserting {count} rows")
            conn.execute(stmt, data)
            conn.commit()  # Ensure changes are committed
            data = []

if data:
    with engine.connect() as conn:
        conn.execute(stmt, data)
        conn.commit()  # Ensure changes are committed

print("Done cleaning text")